# 집계_2_중소기업
위기 지역 수출에서 **중소기업 비중**이 얼마인지, 위기 때 **중소기업과 그 외 기업의 수출 변화**가 어떻게 달랐는지 계산합니다.

- 지역: 홍해 → 유럽 13개국 / 호르무즈 → 걸프 8개국 (참고용으로 유럽·중동 대륙 합계도 계산)
- 입력: `finance/142_DT_B10066_20260930070146.csv` (KOSIS 중소기업수출동향, 국가별 총수출·중소기업 수출, 분기)
- 출력: `집계_2_중소기업.xlsx` (설명 / 중소기업비중_몫 / 위기기간_증가율 / 분기별_증가율 / 지역별_분기수출)

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

KOSIS_FILE = KEYS / "142_DT_B10066_20260930070146.csv"
REGIONS = {
    "유럽13개국": ["독일", "네덜란드", "벨기에", "영국", "프랑스", "폴란드", "스웨덴", "덴마크", "핀란드", "체코공화국",
                "오스트리아", "헝가리", "슬로바키아"],
    "걸프8개국": ["아랍에미리트 연합", "사우디아라비아", "카타르", "쿠웨이트", "오만", "바레인", "이라크", "이란"],
}
CRISES = {   # 위기: (지역, 비중 기준 분기 = 위기 직전 4개 분기, 위기 분기)
    "홍해": ("유럽13개국", ["2022Q4", "2023Q1", "2023Q2", "2023Q3"], ["2024Q1", "2024Q2", "2024Q3", "2024Q4"]),
    "호르무즈": ("걸프8개국", ["2025Q1", "2025Q2", "2025Q3", "2025Q4"], ["2026Q1", "2026Q2"]),
}

## 1. KOSIS 표 정리

In [ ]:
raw = pd.read_csv(KOSIS_FILE, encoding="cp949")
raw.columns = [col.strip() for col in raw.columns]
raw["국가별"] = raw["국가별"].str.strip()
raw["항목"] = raw["항목"].str.replace(" ", "")
qcols = [col for col in raw.columns if "/4" in col]
long = raw.melt(id_vars=["국가별", "항목"], value_vars=qcols, var_name="분기", value_name="금액")
long["분기"] = long["분기"].str.replace(r"(\d{4})\.(\d)/4", r"\1Q\2", regex=True)
long["금액"] = pd.to_numeric(long["금액"].astype(str).str.replace(",", "").str.strip(), errors="coerce")
long = long[long["항목"].isin(["총수출", "중소기업"])].dropna(subset=["금액"])
wide = long.pivot_table(index=["국가별", "분기"], columns="항목", values="금액", aggfunc="sum").reset_index()
wide["그외기업"] = wide["총수출"] - wide["중소기업"]

print("이름이 안 맞는 국가:", {r: sorted(set(c) - set(wide["국가별"])) for r, c in REGIONS.items()})

COLS = ["총수출", "중소기업", "그외기업"]
reg = {r: wide[wide["국가별"].isin(c)].groupby("분기")[COLS].sum() for r, c in REGIONS.items()}
cont = wide.set_index(["국가별", "분기"])
idx = cont.loc["유럽"].index
reg["유럽대륙(러시아·벨라루스 제외)"] = (cont.loc["유럽"][COLS]
                                - cont.loc["러시아 연방"][COLS].reindex(idx, fill_value=0)
                                - cont.loc["벨라루스"][COLS].reindex(idx, fill_value=0))
reg["중동대륙"] = cont.loc["중동"][COLS]
world = cont.loc["계"][COLS]

## 2. 위기 지역의 중소기업 비중 (수출 금액 기준)

In [ ]:
share = []
for crisis, (rname, base_q, _) in CRISES.items():
    r = reg[rname].loc[base_q]
    share.append({"위기": crisis, "지역": rname, "비중기준분기": f"{base_q[0]}~{base_q[-1]}",
                  "총수출_억달러": round(r["총수출"].sum() / 1e8, 1),
                  "중소기업수출_억달러": round(r["중소기업"].sum() / 1e8, 1),
                  "지역중소기업비중_%": round(r["중소기업"].sum() / r["총수출"].sum() * 100, 1),
                  "전체수출중_중소기업비중_%": round(world.loc[base_q, "중소기업"].sum()
                                             / world.loc[base_q, "총수출"].sum() * 100, 1)})
share = pd.DataFrame(share)
share

## 3. 위기 때 수출 증가율: 중소기업 vs 그 외 기업
지역효과 = 위기 지역으로의 증가율 − 그 밖의 지역으로의 증가율 (같은 기업군의 전반적 경기 흐름을 뺀 값)

In [ ]:
def yoy(df, col, qs):
    prev = [f"{int(q[:4]) - 1}{q[4:]}" for q in qs]
    return (df.loc[qs, col].sum() / df.loc[prev, col].sum() - 1) * 100


grow, summ = [], []
for crisis, (rname, _, crisis_q) in CRISES.items():
    rd = reg[rname]
    other = world - rd.reindex(world.index, fill_value=0)
    for label, qs in [(None, crisis_q)] + [(q, [q]) for q in crisis_q]:
        row = {"위기": crisis, "지역": rname}
        row["위기분기" if label is None else "분기"] = f"{qs[0]}~{qs[-1]}" if label is None else label
        row["중소기업_증가율_%"] = yoy(rd, "중소기업", qs)
        row["그외기업_증가율_%"] = yoy(rd, "그외기업", qs)
        k1, k2 = ("다른지역_중소기업_%", "다른지역_그외기업_%") if label is None else \
                 ("다른지역_중소기업_증가율_%", "다른지역_그외기업_증가율_%")
        row[k1] = yoy(other, "중소기업", qs)
        row[k2] = yoy(other, "그외기업", qs)
        row["중소기업_지역효과_%p"] = row["중소기업_증가율_%"] - row[k1]
        row["그외기업_지역효과_%p"] = row["그외기업_증가율_%"] - row[k2]
        row = {kk: (round(v, 1) if isinstance(v, float) else v) for kk, v in row.items()}
        (summ if label is None else grow).append(row)
summ, grow = pd.DataFrame(summ), pd.DataFrame(grow)
summ

## 4. 저장

In [ ]:
notes = pd.DataFrame([
    ("목적", "해협 위기 피해 중 중소기업 몫과, 중소기업이 더 크게 피해를 봤는지 확인"),
    ("자료", "KOSIS 중소기업수출동향(중소벤처기업부, 관세청 통관자료 기반) 국가별 총수출·중소기업 수출, 분기, 달러"),
    ("지역", "유럽13개국 = 독일·네덜란드·벨기에·영국·프랑스·폴란드·스웨덴·덴마크·핀란드·체코·오스트리아·헝가리·슬로바키아 / "
            "걸프8개국 = UAE·사우디·카타르·쿠웨이트·오만·바레인·이라크·이란. 대륙 합계는 참고"),
    ("중소기업 비중", "위기 직전 4개 분기 중소기업 수출 ÷ 총수출 (홍해 2022Q4~2023Q3, 호르무즈 2025Q1~Q4)"),
    ("중소기업 몫", "집계_1의 KCCI 기준 추가 운임 부담 × 중소기업 비중 (수출액 비중 기준 배분)"),
    ("그외기업", "총수출 − 중소기업 수출 (대기업·중견기업·기타)"),
    ("지역효과", "위기 지역으로의 전년비 증가율 − 그 밖의 지역으로의 전년비 증가율. 같은 기업군의 전반적 경기 흐름을 뺀 값"),
    ("주의", "컨테이너 수는 기업 규모별 통계가 없어 수출액 비중으로 배분. 중소기업은 컨테이너당 물건값이 낮은 편이라 "
            "실제 컨테이너 비중은 이보다 높을 가능성(=중소기업 몫을 작게 잡는 쪽)"),
], columns=["항목", "내용"])
out = SAVE / "집계_2_중소기업.xlsx"
with pd.ExcelWriter(out) as w:
    notes.to_excel(w, sheet_name="설명", index=False)
    share.to_excel(w, sheet_name="중소기업비중_몫", index=False)
    summ.to_excel(w, sheet_name="위기기간_증가율", index=False)
    grow.to_excel(w, sheet_name="분기별_증가율", index=False)
    pd.concat(reg, names=["지역", "분기"]).reset_index().to_excel(w, sheet_name="지역별_분기수출", index=False)
print("저장:", out)